# Lithography Ops AI — Stage C: Evaluating the RAG System (Colab)

Stages A & B *built* semantic retrieval and grounded generation. Stage C
**measures** them, turning "it looks right" into numbers you can defend.

We compute standard information-retrieval metrics against a **hand-labelled test
set** (operator-style questions paired with the document that should be found):

- **Hit@1** — how often the *top* result is correct
- **Hit@3** — how often a correct result is in the top 3
- **MRR** (mean reciprocal rank) — rewards putting the right answer near the top

Plus a lightweight **groundedness** check for generated reports (no judge model
needed): flags fabricated citations and invented numbers.

Everything is free, local, no API keys. Run cells top to bottom.

## 1. Install + load knowledge base

In [ ]:
!pip -q install sentence-transformers faiss-cpu
KNOWLEDGE = [
  {
    "doc_id": "KB-OVL-01",
    "subsystem": "reticle stage",
    "title": "Overlay error drift diagnosis",
    "content": "Rising overlay error over several hours commonly indicates reticle stage calibration drift rather than a sudden fault. Begin by reviewing the stage position sensor trend and comparing left/right mark residuals. Re-run the overlay calibration sequence; if residuals persist above 3 nm, inspect the reticle stage sensor (part P-STG-04) for contamination or aging. Confirm the correction model has not saturated. Escalate to a stage specialist if drift resumes within one shift of recalibration."
  },
  {
    "doc_id": "KB-OVL-02",
    "subsystem": "reticle stage",
    "title": "Reticle stage vibration signature analysis",
    "content": "Elevated vibration on the reticle stage typically appears first as increased high-frequency content during acceleration phases. Capture a vibration spectrum and look for peaks near the stage resonance band. Persistent broadband vibration suggests bearing wear or a loose counterbalance; narrow peaks suggest a control-loop tuning issue. Cross-check with overlay residuals, since stage vibration frequently degrades overlay before it trips any alarm."
  },
  {
    "doc_id": "KB-OVL-03",
    "subsystem": "reticle stage",
    "title": "Overlay calibration procedure",
    "content": "Standard overlay recalibration: place the calibration reticle, run the mark detection routine across all field points, and let the system compute the correction grid. Verify the reported model error is within specification before releasing the tool. If the routine fails mark detection repeatedly, the illumination on the alignment sensor may be low; clean the sensor window and retry before replacing hardware."
  },
  {
    "doc_id": "KB-COOL-01",
    "subsystem": "cooling",
    "title": "Cooling system temperature rise",
    "content": "Temperature and focus error rising together is a classic signature of a cooling fault. The thermal expansion from inadequate cooling shifts the focal plane, so focus error tracks the temperature climb. Inspect the pump seals (part P-COOL-01), verify coolant flow rate against the nominal setpoint, and check for air entrainment in the loop. If alarms persist after flow is restored, schedule a maintenance window before the temperature reaches the interlock threshold."
  },
  {
    "doc_id": "KB-COOL-02",
    "subsystem": "cooling",
    "title": "Coolant pump seal replacement",
    "content": "A degraded pump seal shows as a slow decline in coolant flow and occasional pressure oscillation. To replace seal kit P-COOL-01: isolate the loop, relieve pressure, drain to the service level, and swap the seal following the torque sequence. Bleed air from the loop before returning to service and confirm flow stabilizes at setpoint. Log the coolant top-up volume for trend tracking."
  },
  {
    "doc_id": "KB-COOL-03",
    "subsystem": "cooling",
    "title": "Focus error caused by thermal drift",
    "content": "When focus error climbs without any optical fault, suspect thermal drift in the frame or wafer chuck. Confirm by correlating focus error against the temperature sensor: a lag of a few minutes between temperature and focus is expected. Once cooling is restored, focus error should recover within the thermal time constant. If it does not recover, escalate to optics."
  },
  {
    "doc_id": "KB-COOL-04",
    "subsystem": "cooling",
    "title": "Coolant flow interlock troubleshooting",
    "content": "A coolant flow interlock trip halts exposure to protect the system. First verify the flow sensor reading against a manual gauge to rule out a faulty sensor. If flow is genuinely low, check for a clogged filter, a failing pump, or a closed isolation valve. Do not bypass the interlock; restore flow and clear the alarm through the normal reset path."
  },
  {
    "doc_id": "KB-SRC-01",
    "subsystem": "source",
    "title": "Source power degradation",
    "content": "Sagging source power together with throughput loss points to a source module problem rather than a stage or cooling issue. Verify the power module (part P-SRC-02) output against its commanded level and inspect the collector for contamination that reduces transmitted power. A gradual decline usually means collector degradation; a sudden step suggests a module fault."
  },
  {
    "doc_id": "KB-SRC-02",
    "subsystem": "source",
    "title": "Collector contamination cleaning",
    "content": "Collector contamination reduces delivered source power and lowers wafer throughput because dose targets take longer to reach. Follow the collector inspection routine, and if reflectivity is below threshold, schedule the cleaning procedure. After cleaning, re-measure delivered power and update the dose calibration before resuming production."
  },
  {
    "doc_id": "KB-SRC-03",
    "subsystem": "source",
    "title": "Source power module fault isolation",
    "content": "To isolate a source power module fault, compare commanded versus delivered power across a range of setpoints. A consistent offset at all setpoints points to a calibration issue; instability or dropouts point to the module hardware (part P-SRC-02). Replace the module only after confirming cabling and the control signal are healthy."
  },
  {
    "doc_id": "KB-SRC-04",
    "subsystem": "source",
    "title": "Throughput loss root-cause checklist",
    "content": "Wafer throughput loss has several possible roots. Rank them: reduced source power (dose takes longer), stage settling delays, increased alarm-driven pauses, and wafer-handling slowdowns. Check delivered source power first since it is the most common cause, then review the alarm log for repeated brief stoppages that erode throughput without a single obvious fault."
  },
  {
    "doc_id": "KB-VAC-01",
    "subsystem": "vacuum",
    "title": "Vacuum pressure excursion response",
    "content": "A vacuum pressure excursion can disturb both source performance and contamination control. On a pressure rise, check for a leak at recently serviced flanges, verify pump status, and review the outgassing history if a new component was installed. Small slow rises are often outgassing; sharp rises indicate a leak or pump fault."
  },
  {
    "doc_id": "KB-VAC-02",
    "subsystem": "vacuum",
    "title": "Vacuum pump maintenance schedule",
    "content": "Vacuum pumps follow a preventive schedule based on run hours and observed base pressure. Track base pressure over time; a rising trend at constant load signals approaching service need. Perform the scheduled service before base pressure crosses the action limit to avoid an unplanned interruption."
  },
  {
    "doc_id": "KB-VAC-03",
    "subsystem": "vacuum",
    "title": "Leak detection procedure",
    "content": "For suspected vacuum leaks, isolate sections and observe the pressure rate of rise. Use the tracer-gas method around suspect flanges. Document which section shows the fastest rise. Re-torque or reseal the identified flange, then confirm base pressure returns to nominal before releasing the tool."
  },
  {
    "doc_id": "KB-WFR-01",
    "subsystem": "wafer handler",
    "title": "Wafer handler alarm recovery",
    "content": "A single transient wafer-handler alarm that auto-recovers is usually benign, often a sensor debounce or a marginal grip event. Review the handler log for repetition. Isolated events need no action beyond logging; repeated events in the same position indicate a mechanical or sensor problem needing inspection."
  },
  {
    "doc_id": "KB-WFR-02",
    "subsystem": "wafer handler",
    "title": "Wafer chuck contamination",
    "content": "Chuck contamination causes clamping errors and can manifest as focus or overlay noise. Inspect the chuck surface, run the cleaning routine, and verify flatness after cleaning. Persistent clamping errors after cleaning suggest a worn chuck or a vacuum-clamp leak."
  },
  {
    "doc_id": "KB-WFR-03",
    "subsystem": "wafer handler",
    "title": "Robot handoff timing errors",
    "content": "Handoff timing errors between the wafer robot and the chuck slow throughput and can trigger alarms. Check the handoff position calibration and the grip confirmation sensor. Small timing drifts are usually recalibrated in software; repeated grip failures point to worn end-effector pads."
  },
  {
    "doc_id": "KB-ALM-01",
    "subsystem": "general",
    "title": "Alarm flood triage",
    "content": "During an alarm flood, group alarms by subsystem and timestamp rather than reacting to each individually. The earliest alarm in a cluster is usually the root cause and later alarms are consequences. Silence non-safety nuisance alarms only after the root cause is identified, never before."
  },
  {
    "doc_id": "KB-ALM-02",
    "subsystem": "general",
    "title": "Alarm count trend interpretation",
    "content": "A rising alarm-count trend, even below the alert threshold, is an early warning that a subsystem is degrading. Correlate the alarm-count rise with sensor trends: alarms rising alongside temperature suggest cooling; alongside overlay suggest the stage. Use the trend to schedule proactive inspection."
  },
  {
    "doc_id": "KB-PM-01",
    "subsystem": "general",
    "title": "Preventive maintenance planning",
    "content": "Preventive maintenance scheduling balances time-since-maintenance against observed health indicators. A tool well past its nominal interval with a declining health score should be prioritized. Confirm required parts are in stock and a qualified specialist is available before opening a maintenance window to avoid extended downtime."
  },
  {
    "doc_id": "KB-PM-02",
    "subsystem": "general",
    "title": "Time-since-maintenance risk factors",
    "content": "As time since maintenance grows, the probability of drift-related faults increases, particularly on the reticle stage and cooling loop. Treat a high time-since-maintenance value combined with any anomalous sensor trend as an elevated-risk condition warranting earlier intervention."
  },
  {
    "doc_id": "KB-HND-01",
    "subsystem": "general",
    "title": "Shift handover best practices",
    "content": "An effective shift handover separates verified facts from suggested actions. State each open issue, the supporting evidence, the owner, and the current status. Avoid mixing speculation with confirmed observations so the incoming shift can act on facts and evaluate suggestions independently."
  },
  {
    "doc_id": "KB-HND-02",
    "subsystem": "general",
    "title": "Escalation criteria",
    "content": "Escalate when a required specialist is unavailable, a needed part is out of stock with a long lead time, or a health score continues to decline after corrective action. Escalation should include the evidence gathered so the next tier does not repeat the investigation from scratch."
  },
  {
    "doc_id": "KB-FOC-01",
    "subsystem": "cooling",
    "title": "Focus error versus overlay error differentiation",
    "content": "Focus error and overlay error have different root causes and should not be confused. Focus error most often tracks thermal and optical issues, while overlay error tracks stage and alignment issues. When both rise together, thermal drift affecting both the focal plane and stage positioning is a common shared cause worth checking first."
  },
  {
    "doc_id": "KB-THR-01",
    "subsystem": "source",
    "title": "Wafer throughput baseline and deviation",
    "content": "Establish a wafer-throughput baseline per tool and product. A sustained deviation below baseline, once wafer mix is accounted for, indicates a developing problem. Pair throughput deviation with source power and alarm trends to distinguish a source issue from a handling or stage issue."
  },
  {
    "doc_id": "KB-VIB-01",
    "subsystem": "reticle stage",
    "title": "Vibration threshold and early warning",
    "content": "Vibration rising steadily toward its threshold is a reliable early-warning indicator for stage mechanical wear. Trend the vibration RMS; a monotonic rise over multiple shifts warrants inspection before the threshold trips, creating time to plan rather than react."
  },
  {
    "doc_id": "KB-DRIFT-01",
    "subsystem": "general",
    "title": "Slow sensor drift detection",
    "content": "Slow sensor drift is dangerous precisely because it stays below alarm limits until it suddenly does not. Anomaly detection on combinations of sensors catches drift earlier than single-sensor thresholds. When an anomaly model flags drift, review the contributing sensors to localize the subsystem."
  },
  {
    "doc_id": "KB-INC-01",
    "subsystem": "general",
    "title": "Incident prioritization framework",
    "content": "Prioritize incidents by severity and by remaining useful life. A high-severity incident on a machine with low remaining useful life demands immediate attention; a low-severity transient that auto-recovered can be logged and monitored. Always attach the supporting sensor evidence to the incident record so prioritization is traceable."
  },
  {
    "doc_id": "KB-RUL-01",
    "subsystem": "general",
    "title": "Remaining useful life interpretation",
    "content": "A short predicted remaining useful life means intervention should be planned now, while a long value supports normal operation. Treat remaining useful life as a planning aid alongside the health score and failure risk, not as a guarantee. Confirm the prediction against the underlying sensor trends before acting on it."
  }
]
print(f'Loaded {len(KNOWLEDGE)} synthetic documents.')

## 2. Build the retriever (Stages A, condensed)

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
texts = [f"{r['title']}. {r['content']}" for r in KNOWLEDGE]
emb = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)

class Retriever:
    def search(self, query, k=3):
        q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
        s, idx = index.search(q, k)
        return [dict(KNOWLEDGE[i], score=round(float(sc),3)) for sc,i in zip(s[0], idx[0])]

retriever = Retriever()
print('Retriever ready.')

## 3. The labelled test set

Each query is written in plain operator language (not the document wording), so
this genuinely tests *semantic* retrieval. Each is paired with the document id(s)
that count as a correct answer.

In [ ]:
TEST_SET = [
    ('the machine is getting hot and the image is blurry', {'KB-COOL-01','KB-COOL-03','KB-FOC-01'}),
    ('wafers are coming out slower than usual', {'KB-SRC-04','KB-THR-01','KB-SRC-01'}),
    ('how do I hand over open issues to the next shift', {'KB-HND-01'}),
    ('the alignment is slowly getting worse over the day', {'KB-OVL-01','KB-OVL-03','KB-FOC-01'}),
    ('coolant is not flowing and exposure stopped', {'KB-COOL-04','KB-COOL-02'}),
    ('the light source seems weaker than before', {'KB-SRC-01','KB-SRC-02','KB-SRC-03'}),
    ('too many alarms going off at once', {'KB-ALM-01','KB-ALM-02'}),
    ('the stage is shaking during moves', {'KB-OVL-02','KB-VIB-01'}),
    ('pressure in the vacuum chamber is climbing', {'KB-VAC-01','KB-VAC-02','KB-VAC-03'}),
    ('the robot keeps dropping or mis-handling wafers', {'KB-WFR-01','KB-WFR-03','KB-WFR-02'}),
    ('when should we do preventive maintenance', {'KB-PM-01','KB-PM-02'}),
    ('how long until this machine fails', {'KB-RUL-01','KB-DRIFT-01'}),
]
print(len(TEST_SET), 'labelled test queries.')

## 4. Compute retrieval metrics

In [ ]:
def evaluate_retrieval(retriever, k=3):
    hits1 = hits3 = 0; rr_sum = 0.0; rows = []
    for query, correct in TEST_SET:
        ids = [r['doc_id'] for r in retriever.search(query, k=k)]
        h1 = bool(ids) and ids[0] in correct
        h3 = any(i in correct for i in ids)
        rr = 0.0
        for rank, i in enumerate(ids, 1):
            if i in correct: rr = 1.0/rank; break
        hits1 += h1; hits3 += h3; rr_sum += rr
        rows.append((query, ids[0] if ids else '-', h1, h3, round(rr,2)))
    n = len(TEST_SET)
    return {'n': n, 'hit@1': round(hits1/n,3), 'hit@3': round(hits3/n,3),
            'MRR': round(rr_sum/n,3)}, rows

summary, rows = evaluate_retrieval(retriever, k=3)
print('=== RETRIEVAL EVALUATION ===')
for k,v in summary.items(): print(f'  {k}: {v}')
print('\nPer-query (query | top result | hit@1 | hit@3 | RR):')
for q, top, h1, h3, rr in rows:
    print(f"  {('OK ' if h1 else '   ')}{top:12} rr={rr}  <- {q}")

## 5. Interpreting the numbers

- **Hit@3 near 1.0** means the right document is almost always retrieved — the
  system reliably finds relevant knowledge.
- **Hit@1** is stricter (top result exactly right). Slightly lower is normal and
  fine, because several documents can be legitimately relevant to one symptom.
- **MRR** close to 1.0 means correct answers sit at or near the top.

For an interview: *"I evaluated retrieval on a labelled test set and got Hit@3 of
X and MRR of Y, so the semantic search reliably surfaces the right procedure."*

## 6. Groundedness check (for generated reports)

A quick, judge-model-free check that a report didn't hallucinate: every cited
document must have actually been retrieved, and every number in the report must
have been in the supplied facts.

In [ ]:
import re
_NUM = re.compile(r'\d+\.?\d*')
def groundedness(report_text, allowed_numbers, retrieved_ids, cited_ids):
    fabricated = sorted(set(cited_ids) - set(retrieved_ids))
    cleaned = re.sub(r'KB-[A-Z]+-\d+', ' ', report_text)
    invented = sorted(set(_NUM.findall(cleaned)) - set(allowed_numbers))
    return {'grounded': not fabricated and not invented,
            'fabricated_citations': fabricated, 'invented_numbers': invented}

# Example: a good grounded report vs a hallucinated one
good = groundedness('Health 7.2, cooling fault. See KB-COOL-01.',
                    {'7.2'}, {'KB-COOL-01','KB-COOL-03'}, {'KB-COOL-01'})
bad  = groundedness('Health 45, wait 3 hours. See KB-FAKE-99.',
                    {'7.2'}, {'KB-COOL-01'}, {'KB-FAKE-99'})
print('Good report:', good)
print('Bad report :', bad)

## What you just built

A real **evaluation harness** for your RAG system: labelled retrieval metrics
(Hit@1, Hit@3, MRR) and a groundedness check for generated reports. This is the
difference between "I used AI" and "I measured that my AI works" — exactly what a
technical interviewer wants to see.

**Next (Stage D):** observability — trace every run end to end (which agent,
which tool, what was retrieved, timings) so the whole pipeline is auditable.